**Software Name:** ib-ORAN
<br>
**SPDX-FileCopyrightText:** Copyright (c) Orange SA
<br>
**SPDX-License-Identifier:** MIT
 
This software is distributed under the MIT license,
<br>
see the [LICENSE](../LICENSE) file for more details
<br>
or https://spdx.org/licenses/MIT.html
 
Authors: see [CONTRIBUTORS.md](../CONTRIBUTORS.md)

# Simulation Pipeline

### Path Setup

In [ ]:
import sys
import os

sys.path.append(os.path.abspath(".."))
IBORAN_DIR = os.path.abspath("..")
IBORAN_RUN_DIR = os.path.abspath("../model/iboran_campaign")

run_env = os.environ.copy()
run_env["PYTHONPATH"] = IBORAN_DIR + ":" + run_env.get("PYTHONPATH", "")

save_dir = "results"

### Font Setup

In [ ]:
import os, urllib.request
import matplotlib as mpl
from matplotlib import font_manager

font_dir = os.path.expanduser("~/.fonts/poppins")
os.makedirs(font_dir, exist_ok=True)

base = "https://github.com/google/fonts/raw/main/ofl/poppins/"
for w in ["Regular", "Medium", "SemiBold", "Bold", "Italic"]:
    path = os.path.join(font_dir, f"Poppins-{w}.ttf")
    if not os.path.exists(path):
        urllib.request.urlretrieve(base + f"Poppins-{w}.ttf", path)
    font_manager.fontManager.addfont(path)

mpl.rcParams.update(
    {
        "font.family": ["Poppins", "DejaVu Sans"],  # DejaVu as glyph fallback
        "mathtext.fontset": "dejavusans",
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
    }
)

### Campaign Configuration

In [ ]:
from model.iboran_campaign.iboran_campaign import IbOranCampaign

mc_iterations = 1  # number of Monte Carlo iterations

dataset_id = "market-1501"  # "toy" | "market-1501"
data_format = "EMBEDDING"  # "IMAGE" | "VIDEO" | "EMBEDDING" | "TEXT"
embedder = "TransReID"  # "TransReID" | "CLIP"
oid_range = [4, 5]  # [low, high], inclusive range of object IDs
save_dapp_log = False
save_learning_convg = False
verbose = False

campaign = IbOranCampaign(
    run_cwd=IBORAN_RUN_DIR,
    run_env=run_env,
    mc_iterations=mc_iterations,
    dataset_id=dataset_id,
    data_format=data_format,
    embedder=embedder,
    oid_range=oid_range,
    save_dapp_log=save_dapp_log,
    save_learning_convg=save_learning_convg,
    verbose=verbose,
)

### Sweep Parameters

In [ ]:
from model.iboran_interfaces.iboran_shm_protocol import RatType

dom_rat_type = [RatType.NR]  # RatType.NR | RatType.LTE

# dApp control period [subframes]
dom_dapp_control_period_sf = [1, 2, 5, 10]
base_dapp_control_period_sf = 1

# xApp control period [frames]
dom_xapp_control_period_f = [1, 10]
base_xapp_control_period_f = 10

# max parallel UEs scheduled per TTI/slot
dom_k = [1, 2, 4, 6]
base_k = 2

# dApp algos (see doc/built-in-algos.md)
dom_dapp_algo_ue_selection = [
    # "ibRandom",
    "Random",
    "ibRoundRobin",
    "RoundRobin",
    # "ibTopKCqi",
    # "ibTopKBufferLevel",
    # "ibTopKPropFairness",
    # "ibTopKCriticality",
    # "ibTopKMultiCriteria",
    "TopKCqi",
    "TopKBufferLevel",
    "TopKPropFairness",
    "TopKCriticality",
    "TopKMultiCriteria",
    # "ibGreedyCqi",
    # "ibGreedyBufferLevel",
    # "ibGreedyPropFairness",
    # "ibGreedyCriticality",
    # "ibGreedyMultiCriteria",
    "GreedyCqi",
    "GreedyBufferLevel",
    "GreedyPropFairness",
    "GreedyCriticality",
    "GreedyMultiCriteria",
    # "ibDpCqi",
    # "ibDpBufferLevel",
    # "ibDpPropFairness",
    # "ibDpCriticality",
    # "ibDpMultiCriteria",
    "DpCqi",
    "DpBufferLevel",
    "DpPropFairness",
    "DpCriticality",
    "DpMultiCriteria",
]
base_dapp_algo_ue_selection = "RoundRobin"

# xApp algos (see doc/built-in-algos.md)
dom_xapp_algo_rbg_bounding = [
    "Bypass",
    "Clamp",
]
base_xapp_algo_rbg_bounding = "Bypass"

### Execute Campaign

In [ ]:
df = campaign.run(
    dom_rat_type=dom_rat_type,
    dom_dapp_control_period_sf=dom_dapp_control_period_sf,
    dom_xapp_control_period_f=dom_xapp_control_period_f,
    dom_k=dom_k,
    dom_dapp_algo_ue_selection=dom_dapp_algo_ue_selection,
    dom_xapp_algo_rbg_bounding=dom_xapp_algo_rbg_bounding,
)

# Evaluation Pipeline

### KPI Selection

In [ ]:
kpis = [
    "SatisfactionScore",
    "PDR",
    "Throughput",
    "Latency",
    "PRBUtilizationRatio",
    "DAppDecisionTime",
    "XAppDecisionTime",
]

# from model.iboran_campaign.iboran_orchestrator import _BASE_KPI_TEMPLATES
# kpis = list(_BASE_KPI_TEMPLATES.keys())

## dApp Algorithms: UE Selection

In [ ]:
per_algo = "dapp_algo_ue_selection"

### KPI vs. Maximum Parallel UEs

In [ ]:
campaign.plot_kpi_vs_param_per_algo(
    against="k",
    per=per_algo,
    which_kpi=kpis,
    which_dapp_control_period_sf=base_dapp_control_period_sf,
    which_xapp_control_period_f=base_xapp_control_period_f,
    which_k=dom_k,
    which_dapp_algo_ue_selection=dom_dapp_algo_ue_selection,
    which_xapp_algo_rbg_bounding=base_xapp_algo_rbg_bounding,
    which_rat=dom_rat_type,
    save_dir=save_dir,
)

### KPI vs. dApp Control Period

In [ ]:
campaign.plot_kpi_vs_param_per_algo(
    against="dapp_control_period_sf",
    per=per_algo,
    which_kpi=kpis,
    which_dapp_control_period_sf=dom_dapp_control_period_sf,
    which_xapp_control_period_f=base_xapp_control_period_f,
    which_k=base_k,
    which_dapp_algo_ue_selection=dom_dapp_algo_ue_selection,
    which_xapp_algo_rbg_bounding=base_xapp_algo_rbg_bounding,
    which_rat=dom_rat_type,
    save_dir=save_dir,
)

### Aggregate KPI Distribution

In [ ]:
campaign.plot_bar_kpi_per_algo(
    per=per_algo,
    which_kpi=kpis,
    which_dapp_algo_ue_selection=dom_dapp_algo_ue_selection,
    which_xapp_algo_rbg_bounding=base_xapp_algo_rbg_bounding,
    which_rat=dom_rat_type,
    save_dir=save_dir,
)

### Baseline Comparison

In [ ]:
dom_ref_algo = [
    "ibRoundRobin",
    "ibGreedyCqi",
    "ibGreedyBufferLevel",
    "ibGreedyPropFairness",
    "ibGreedyCriticality",
]

for ref_algo in dom_ref_algo:
    campaign.plot_table_gain_loss_kpi_per_algo(
        per=per_algo,
        which_kpi=kpis,
        which_dapp_algo_ue_selection=dom_dapp_algo_ue_selection,
        which_xapp_algo_rbg_bounding=base_xapp_algo_rbg_bounding,
        which_rat=dom_rat_type,
        ref_algo=ref_algo,
    )

### KPI Summary

In [ ]:
campaign.plot_table_kpi_per_algo(
    per=per_algo,
    which_kpi=kpis,
    which_dapp_algo_ue_selection=dom_dapp_algo_ue_selection,
    which_xapp_algo_rbg_bounding=base_xapp_algo_rbg_bounding,
    which_rat=dom_rat_type,
)

## xApp Algorithms: RBG Bounding

In [ ]:
per_algo = "xapp_algo_rbg_bounding"

### KPI vs. Maximum Parallel UEs

In [ ]:
campaign.plot_kpi_vs_param_per_algo(
    against="k",
    per=per_algo,
    which_kpi=kpis,
    which_dapp_control_period_sf=base_dapp_control_period_sf,
    which_xapp_control_period_f=base_xapp_control_period_f,
    which_k=dom_k,
    which_dapp_algo_ue_selection=base_dapp_algo_ue_selection,
    which_xapp_algo_rbg_bounding=dom_xapp_algo_rbg_bounding,
    which_rat=dom_rat_type,
    save_dir=save_dir,
)

### KPI vs. xApp Control Period

In [ ]:
campaign.plot_kpi_vs_param_per_algo(
    against="xapp_control_period_f",
    per=per_algo,
    which_kpi=kpis,
    which_dapp_control_period_sf=base_dapp_control_period_sf,
    which_xapp_control_period_f=dom_xapp_control_period_f,
    which_k=base_k,
    which_dapp_algo_ue_selection=base_dapp_algo_ue_selection,
    which_xapp_algo_rbg_bounding=dom_xapp_algo_rbg_bounding,
    which_rat=dom_rat_type,
    save_dir=save_dir,
)

### Aggregate KPI Distribution

In [ ]:
campaign.plot_bar_kpi_per_algo(
    per=per_algo,
    which_kpi=kpis,
    which_dapp_algo_ue_selection=base_dapp_algo_ue_selection,
    which_xapp_algo_rbg_bounding=dom_xapp_algo_rbg_bounding,
    which_rat=dom_rat_type,
    save_dir=save_dir,
)

### Baseline Comparison

In [ ]:
dom_ref_algo = [
    "Bypass",
    "ibLearningPPO",
    "LearningPPO",
]

for ref_algo in dom_ref_algo:
    campaign.plot_table_gain_loss_kpi_per_algo(
        per=per_algo,
        which_kpi=kpis,
        which_dapp_algo_ue_selection=base_dapp_algo_ue_selection,
        which_xapp_algo_rbg_bounding=dom_xapp_algo_rbg_bounding,
        which_rat=dom_rat_type,
        ref_algo=ref_algo,
    )

### KPI Summary

In [ ]:
campaign.plot_table_kpi_per_algo(
    per=per_algo,
    which_kpi=kpis,
    which_dapp_algo_ue_selection=base_dapp_algo_ue_selection,
    which_xapp_algo_rbg_bounding=dom_xapp_algo_rbg_bounding,
    which_rat=dom_rat_type,
)